In [ ]:
#!pip install datasets

In [ ]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import tools.ai_token as tk
import tools.ai_attention_decoder as att

In [63]:
ds = load_dataset("csebuetnlp/xlsum", "french")

README.md: 0.00B [00:00, ?B/s]

xlsum.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found xlsum.py

In [2]:
from datasets import load_dataset

# On utilise un dataset qui n'a PAS de fichier .py dans son repo
# 'asi/pagnol' est un excellent corpus français au format Parquet
try:
    dataset = load_dataset("asi/pagnol", "pagnol_small") # Version small pour tester
    print("Succès ! Dataset PAGnol chargé.")
except Exception as e:
    print(f"Échec PAGnol. Tentative sur une version statique de Wiki...")
    # Autre option : un dataset qui est déjà en pur Parquet sur le Hub
    dataset = load_dataset("tatsu-lab/alpaca", split="train") # Souvent utilisé comme test universel

# Vérification du contenu
print(f"Colonnes : {dataset.column_names}")
print(f"Exemple : {dataset[0]['text'][:200]}...")

Échec PAGnol. Tentative sur une version statique de Wiki...
Colonnes : ['instruction', 'input', 'output', 'text']
Exemple : Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Response:
1.Eat a balanced diet an...


In [3]:
print(dataset)

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 52002
})


In [4]:
# Extraction des textes du dataset Alpaca
# On prend la colonne 'text' qui contient l'instruction et la réponse
corpus = [ex['text'] for ex in dataset.select(range(10000))]
full_text = " ".join(corpus)

# Lance ton entraînement BPE ici
# token.train_optimiser_v2(full_text)

In [5]:
token = tk.BPETokenizer('data/corpus_francais.txt',10)

Nettoyage terminé:


In [6]:
token.text_cleaned = full_text
token.addToken = 20000;

In [60]:
print(token.text_cleaned)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Response:
1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule. Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What are the three primary colors?

### Response:
The three primary colors are red, blue, and yellow. Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Describe the structure of an atom.

### Response:
An atom is made up of a nucleus, which contains protons and neutrons, surrounded by electrons that travel in orbits around the nucleus. The protons and neutrons have a positive charge, while the electrons have a negative 

In [7]:
token.train_optimiser_v2()

array([7496,   -1, 1009, ...,   -1,   46,   -1], dtype=int32)

In [8]:
token.display_token()


✓ 500 tokens affichés avec couleurs aléatoires


In [9]:
token._print_stats()


Statistiques Nettoyage
..............................
  Caractères: 10,037,682 → 9,895,452 (98.6%)

Statistiques BPE
..............................
Tokens avant BPE:        10,135,521
Tokens après BPE:        2,394,005
Compression:             4.23x
Tokens uniques:          19456
Merges effectués:        20000

Statistiques Temps
..............................
Encodage effectué:       0.000s
Fast Encodage effectué:  0.000s
Décodage effectué:       0.000s
Train effectué:          0.000s
Train Optimisé effectué: 1284.925s



# Decoder-Only to Transformer

In [11]:
config = { 'n_embd': 16,
            'num_heads': 2,
            'n_layers': 1,
            'block_size': 256,
            'dropout': 0.2,      
        }
parameters = {
    'batch_size': 32,
    'max_iters': 4000,
    'eval_interval': 500,
    'eval_iters': 200,
    'learning_rate': 1e-4
}
prompt = "Ceci est l'histoire d'un ROI, grand et beau qui raconte sa dernière bataille"

In [35]:
from dataclasses import dataclass
import torch

@dataclass
class TransformerConfig:
    # Dimensions du modèle
    vocab_size: int    # Taille de ton dictionnaire (ex: 5000)
    n_embd: int        # Dimension des vecteurs (ex: 384)
    num_heads: int        # Nombre de têtes d'attention (ex: 6)
    n_layers: int       # Nombre de blocs (ex: 6)
    
    # Séquences
    block_size: int    # Longueur max article/résumé (ex: 256)
    dropout: float     # Anti-surapprentissage (ex: 0.1)
    
    # Entraînement
    batch_size: int    # Nombre d'exemples par itération (ex: 32 ou 64)
    learning_rate: float          # Vitesse d'apprentissage (ex: 3e-4)
    max_iters: int     # Nombre total d'itérations
    eval_interval: int # Fréquence des logs
    
    # Matériel
    device: str        # 'mps' pour ton Mac M4 Pro

In [36]:
# 1. On crée la configuration
config = TransformerConfig(
    vocab_size=len(token.vocab),
    n_embd=32,
    num_heads=2,
    n_layers=1,
    block_size=32,
    dropout=0.1,
    batch_size=32, # Tu as 64Go de RAM, profites-en !
    learning_rate=3e-4,
    max_iters=4000,
    eval_interval=500,
    device='mps'
)

# 2. On instancie le modèle en passant les arguments de config
kernel = att.BigramLanguageModeler(
    vocab_size=config.vocab_size,
    n_embd=config.n_embd,
    block_size=config.block_size,
    num_heads=config.num_heads,
    n_layers=config.n_layers,
    dropout=config.dropout,
    device=config.device
).to(config.device)

In [14]:
# Conversion des données en tensor directement sur le device
data = torch.tensor(token.ids, dtype=torch.long, device='mps')
# Split Train/Val
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [40]:
#kernel = att.BigramLanguageModeler(len(token.vocab),'mps',**config)
runner = att.Model_(kernel,token,train_data=train_data,val_data=val_data,config=config)

In [41]:
print(runner.train())

Iter 0: loss 10.0930
Iter 500: loss 7.2579
Iter 1000: loss 7.0969
Iter 1500: loss 6.9276
Iter 2000: loss 6.7201
Iter 2500: loss 6.5090
Iter 3000: loss 6.5072
Iter 3500: loss 6.7150
None


In [59]:
context = torch.tensor([token.encode_np(prompt)], dtype=torch.long, device=config.device)
#context2 = torch.zeros((1, 1), dtype=torch.long, device=config.device)
kernel.eval()
reponse = kernel.generate(context,105)[0]
print(token.decode(reponse.tolist()))

Ceci est l'histoire d'un ROI, grand et beau qui raconte sa dernière bataille, encore que vous destrois tout dont la route-c.

Pour in percé rondequiète vont restèrent, çarières.

"Ce-- qual n'y en lui lui-même, on
lors était donc à sale surtout,
Nous trouvez, et, sont y'abbé nous échelles
porte, à. En, au milieu, il se là-ce parler, se fût que
monsieur fut pât; le quatre
frapp, lui ch imbécileet de réfugier. scandale de jicile jour-moi.

Maisante?Et qui le
disait ignorant;


# Old version

In [ ]:
config = { 'n_embd': 16,
            'num_heads': 2,
            'n_layers': 1,
            'block_size': 256,
            'dropout': 0.2,      
        }
parameters = {
    'batch_size': 32,
    'max_iters': 4000,
    'eval_interval': 500,
    'eval_iters': 200,
    'learning_rate': 1e-4
}
decoder = att.Model(token.ids, token, **config, **parameters, fileout='data/test_3.pth')
decoder.get_model_size()

In [ ]:
decoder.train()
decoder.plot_losses()

In [ ]:
prompt = "Ceci est l'histoire d'un ROI, grand et beau qui raconte sa dernière bataille"
ids_p = token.encode(prompt)
print(ids_p)

In [ ]:
decoder.genere(prompt=prompt,new_token=100)